# TSPN — Full Model Training on Colab (T4 GPU)

Runs `src/training/train.py`'s 6-fold Leave-One-Event-Out cross-validation for the full TSPN model, on a free Colab GPU instead of a local CPU. See `PROJECT_STATE.md` in the repo for why: this project's own local runs measured ~9.5 min/epoch/fold on an 8-core CPU with no usable GPU, while the original plan assumed ~25 min *per fold* on a T4.

**Before running**: Runtime → Change runtime type → **T4 GPU**.

**What you need to upload to Google Drive first** (see the cell below for the exact folder layout):
1. The 6 files in `data/pyg_datasets/*.pt` from your local repo (~192MB total) — **required**, training cannot run without these.
2. *(Optional but recommended)* The contents of `models/checkpoints/` from your local repo, if you have any — this lets Colab **resume** folds that already made progress locally instead of retraining them from scratch. If you skip this, all 6 folds just train fresh, which is still fast on a T4.

Both `data/pyg_datasets/` and `models/checkpoints/` are gitignored (regenerated/large binary artifacts), so they are **not** in the GitHub repo this notebook clones — that's why they need a separate upload.

In [ ]:
# 1. Confirm a GPU is actually attached
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

In [ ]:
# 2. Mount Google Drive (for uploading data once, and persisting checkpoints across sessions)
from google.colab import drive
drive.mount('/content/drive')

### Upload your data now

In the Colab file browser (left sidebar) or directly in Google Drive (drive.google.com), create this folder if it doesn't exist, then upload into it:

```
My Drive/
  tspn_colab/
    pyg_datasets/          <- upload all 6 .pt files from your local data/pyg_datasets/ here
    checkpoints/            <- (optional) upload everything from your local models/checkpoints/ here
```

Uploading 192MB through the Drive web UI can take a few minutes depending on your connection — let it finish before running the next cell.

In [ ]:
# 3. Verify the upload landed where expected
import os
DRIVE_ROOT = '/content/drive/MyDrive/tspn_colab'
pyg_dir = f'{DRIVE_ROOT}/pyg_datasets'
ckpt_dir = f'{DRIVE_ROOT}/checkpoints'
os.makedirs(pyg_dir, exist_ok=True)
os.makedirs(ckpt_dir, exist_ok=True)

pt_files = sorted(f for f in os.listdir(pyg_dir) if f.endswith('.pt'))
print(f'Found {len(pt_files)}/6 PyG dataset files in {pyg_dir}:')
for f in pt_files:
    print(' ', f)
assert len(pt_files) == 6, (
    'Expected exactly 6 .pt files (one per event) in '
    f'{pyg_dir} -- upload them before continuing.'
)

ckpt_files = sorted(os.listdir(ckpt_dir))
print(f'\nFound {len(ckpt_files)} checkpoint/resume-state files in {ckpt_dir} '
      f'({"will resume these folds" if ckpt_files else "none uploaded -- all 6 folds will train fresh"})')

In [ ]:
# 4. Clone the repo (code only -- data/checkpoints come from Drive, not git)
REPO_URL = 'https://github.com/AmalRaj04/tspn.git'
%cd /content
!rm -rf tspn
!git clone $REPO_URL
%cd /content/tspn
!git log --oneline -5

In [ ]:
# 5. Point the repo's data/checkpoint folders at Drive via symlinks --
# no code changes needed, config.py's existing relative paths just work.
import os
os.makedirs('/content/tspn/data', exist_ok=True)
os.makedirs('/content/tspn/models', exist_ok=True)

!rm -rf /content/tspn/data/pyg_datasets
!ln -s {pyg_dir} /content/tspn/data/pyg_datasets

!rm -rf /content/tspn/models/checkpoints
!ln -s {ckpt_dir} /content/tspn/models/checkpoints

# results/tables (attention .npy + train_log CSVs + all_results.csv) also
# persisted to Drive so they survive a session disconnect.
results_dir = f'{DRIVE_ROOT}/results_tables'
os.makedirs(results_dir, exist_ok=True)
!mkdir -p /content/tspn/results
!rm -rf /content/tspn/results/tables
!ln -s {results_dir} /content/tspn/results/tables

!ls -la /content/tspn/data/pyg_datasets/ | head
!ls -la /content/tspn/models/checkpoints/ | head

In [ ]:
# 6. Install dependencies matching Colab's PRE-INSTALLED torch/CUDA
# (do not reinstall torch itself -- Colab's build is already matched to its GPU driver)
import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_TAG = ('cu' + torch.version.cuda.replace('.', '')) if torch.cuda.is_available() else 'cpu'
print(f'torch {TORCH_VERSION}, cuda tag {CUDA_TAG}')

!pip install -q torch_geometric
!pip install -q torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_TAG}.html
!pip install -q pandas pyarrow statsmodels scikit-learn

In [ ]:
# 7. Sanity check: torch_scatter actually works (the same CP06 check used
# throughout this project) before spending any GPU time on real training.
import torch, torch_scatter
x = torch.tensor([1., 2., 3.])
idx = torch.tensor([0, 0, 1])
result = torch_scatter.scatter_add(x, idx)
assert torch.allclose(result, torch.tensor([3., 3.])), f'scatter wrong: {result}'
print('torch_scatter OK')
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 8. Train. Streams progress live; safe to re-run this cell if the session
# disconnects and you reconnect -- train.py resumes automatically from
# whatever is in the Drive-linked checkpoints folder (see PROJECT_STATE.md
# finding #32 for how resume correctness was verified).
%cd /content/tspn
!python src/training/train.py

In [ ]:
# 9. Once all 6 folds finish, inspect the results table (already persisted to Drive via the symlink in step 5)
import pandas as pd
df = pd.read_csv('/content/tspn/results/tables/all_results.csv')
tspn_rows = df[df['model_name'] == 'TSPN']
print(f'{len(tspn_rows)}/6 TSPN folds recorded')
tspn_rows

### Bringing results back to your local machine

Everything under `My Drive/tspn_colab/` (checkpoints, attention weights, train logs, `all_results.csv`) is already the authoritative copy — download that folder from Drive and drop its contents into your local repo's `models/checkpoints/` and `results/tables/` to continue Phase 9/10 work locally, or just keep working from Colab for the ablations too.